# Tier-2 Halo-Ensemble Forecast Tutorial (DK14)

This notebook teaches a reproducible Tier-2 workflow for the `sidm_dsigma` pipeline:

1. Build DK14-based Tier-2 configs for cluster (`HMF`) and dwarf (`SHMR`) ensembles.
2. Run the ensemble forecast script.
3. Inspect `Delta chi^2` summary tables.
4. Display the generated `*_ensemble_summary.png` figures.

Notes:
- Tier-2 means SIDM inner profile + DK14-like outer profile stitching.
- Tier-3 empirical outskirts correction is kept off in this tutorial.


## 1. Setup

Run this notebook from the repository root so `scripts/run_ensemble_forecast.py` and `docs/*.yaml` are discoverable.


In [ ]:
from pathlib import Path
import subprocess
import yaml
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

repo_root = Path.cwd()
assert (repo_root / "scripts" / "run_ensemble_forecast.py").exists(), (
    "Open this notebook with the repository root as the working directory."
)

output_root = repo_root / "outputs" / "notebook_tier2_demo"
config_dir = output_root / "configs"
config_dir.mkdir(parents=True, exist_ok=True)

output_root


## 2. Build Tier-2 DK14 configs

The helper below copies baseline cluster/dwarf YAML files and enforces:
- `tier2.enabled = true`
- `tier2.outer_profile_model = "dk14_like"`
- `tier3.enabled = false`

For quick interactive runs, `quick_demo=True` reduces `N_halos`.


In [ ]:
def config_label(config_path: Path) -> str:
    return config_path.stem.replace("_ensemble_config", "")


def make_tier2_dk14_config(
    base_config_path: Path,
    output_name: str,
    quick_demo: bool = True,
) -> Path:
    config = yaml.safe_load(base_config_path.read_text())

    config.setdefault("tier2", {})
    config.setdefault("tier3", {})

    config["tier2"]["enabled"] = True
    config["tier2"]["outer_profile_model"] = "dk14_like"
    config["tier3"]["enabled"] = False

    if quick_demo:
        if str(config.get("mode", "")).upper() == "HMF":
            config["ensemble"]["N_halos"] = 80
        else:
            config["ensemble"]["N_halos"] = 160

    output_path = config_dir / output_name
    output_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    return output_path


cluster_config = make_tier2_dk14_config(
    base_config_path=repo_root / "docs" / "cluster_ensemble_config.yaml",
    output_name="cluster_tier2_dk14_ensemble_config.yaml",
    quick_demo=True,
)

dwarf_config = make_tier2_dk14_config(
    base_config_path=repo_root / "docs" / "dwarf_ensemble_config.yaml",
    output_name="dwarf_tier2_dk14_ensemble_config.yaml",
    quick_demo=True,
)

cluster_config, dwarf_config


## 3. Run the Tier-2 ensemble forecasts

This executes `scripts/run_ensemble_forecast.py` once for each config.


In [ ]:
def run_forecast(config_path: Path, run_name: str) -> Path:
    run_output_root = output_root / run_name
    run_output_root.mkdir(parents=True, exist_ok=True)
    command = [
        "python",
        "scripts/run_ensemble_forecast.py",
        "--config-path",
        str(config_path),
        "--output-root",
        str(run_output_root),
    ]
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)
    return run_output_root


cluster_output = run_forecast(cluster_config, "cluster")
dwarf_output = run_forecast(dwarf_config, "dwarf")

cluster_output, dwarf_output


## 4. Load and inspect summary tables

Each run writes `tables/<label>_ensemble_delta_chi2_summary.csv` with Tier-1 and Tier-2 rows.


In [ ]:
def summary_table_path(run_output_root: Path, config_path: Path) -> Path:
    label = config_label(config_path)
    return run_output_root / "tables" / f"{label}_ensemble_delta_chi2_summary.csv"


cluster_summary = pd.read_csv(summary_table_path(cluster_output, cluster_config))
dwarf_summary = pd.read_csv(summary_table_path(dwarf_output, dwarf_config))

selected_columns = [
    "tier",
    "sigma_over_m_cm2_g",
    "delta_chi2_optimistic_5pct",
    "required_uniform_percent_for_3sigma",
    "max_abs_delta_sigma_ratio_shift_percent",
]

print("Cluster summary:")
display(cluster_summary[selected_columns])
print("Dwarf summary:")
display(dwarf_summary[selected_columns])


## 5. Show the generated summary figures

The summary figure combines:
- stacked `rho(r)` in `h^2 Msun / pc^3`
- stacked `DeltaSigma(R)` in `h Msun / pc^2`
- SIDM/CDM ratio
- `Delta chi^2` panel

Inset panels show halo-mass and concentration distributions, with mean and 16-84% ranges.


In [ ]:
def summary_figure_path(run_output_root: Path, config_path: Path) -> Path:
    label = config_label(config_path)
    return run_output_root / "figures" / f"{label}_ensemble_summary.png"


def tier_comparison_figure_path(run_output_root: Path, config_path: Path) -> Path:
    label = config_label(config_path)
    return run_output_root / "figures" / f"{label}_ensemble_tier1_vs_tier2_stacked.png"


def show_png(path: Path, title: str) -> None:
    image = plt.imread(path)
    plt.figure(figsize=(10.5, 7.2))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()


show_png(summary_figure_path(cluster_output, cluster_config), "Cluster Tier-2 DK14 Summary")
show_png(tier_comparison_figure_path(cluster_output, cluster_config), "Cluster Tier-1 vs Tier-2 Comparison")
show_png(summary_figure_path(dwarf_output, dwarf_config), "Dwarf Tier-2 DK14 Summary")
show_png(tier_comparison_figure_path(dwarf_output, dwarf_config), "Dwarf Tier-1 vs Tier-2 Comparison")


## 6. Next steps

- Set `quick_demo=False` in the config builder to run full-size ensembles.
- Swap in redshift-split config files in `docs/` to study redshift overlays.
- Change only `tier2.outer_profile_model` if you want to compare DK14-like and other supported outer-profile backends.
